# 🛡️ SecureOps Assistant — Modular RAG System Demo

This notebook demonstrates the retrieval, grounded generation, and evaluation workflow for the **SecureOps Assistant**, an OT/ICS cybersecurity RAG project.

### 🌟 Key Features
1. **Semantic & Keyword Hybrid Search**: Combines ChromaDB dense vector retrieval (`BAAI/bge-small-en-v1.5`) with BM25 keyword search.
2. **Cross-Encoder Reranking**: Uses `cross-encoder/ms-marco-MiniLM-L-6-v2` to select the top 5 most relevant context chunks.
3. **Grounded Generation & Citation**: Uses DeepSeek with strict context grounding and stable CISA, CVE, ATT&CK, and document citations.
4. **Honest Rejection**: Computes confidence scores via a sigmoid function on the top candidate rerank score. Automatically rejects queries outside the knowledge base if confidence is too low.

---

## 🛠️ Step 1: Environment Setup

Install the project dependencies for parsing, indexing, retrieval, and generation.

In [ ]:
# Install dependencies
!pip install -q pymupdf4llm pymupdf rank_bm25 chromadb sentence-transformers openai pandas openpyxl

## 📂 Step 2: Clone Repository & Data Ingestion

Colab needs access to the repository code and knowledge base documents (NIST PDFs and CISA CSAF JSON advisories).

In [ ]:
# Clone repository and cd into workspace
# Replace YOUR_USERNAME after publishing the repository.
!git clone https://github.com/YOUR_USERNAME/secureops-ics-rag.git
%cd secureops-ics-rag

## 🔑 Step 3: Configure DeepSeek API Key

Enter the DeepSeek key using `getpass` so it is not stored in notebook output.

In [ ]:
import os
import getpass

api_key = getpass.getpass("Enter your DEEPSEEK_API_KEY: ")
if api_key.strip():
    os.environ["DEEPSEEK_API_KEY"] = api_key.strip()
    print("DEEPSEEK_API_KEY configured successfully!")
else:
    print("Warning: DEEPSEEK_API_KEY not set; generation will be unavailable.")

## 🗂️ Step 4: Index the Knowledge Base

Build the vector and keyword indexes. We support **Quick Mode** (parses a subset of pages for fast iteration) and **Full Mode** (parses the entire NIST PDFs and all 200 CISA CSAF advisories).

In [ ]:
from src.indexing import build_index

mode = input("Rebuild type (quick/full) [default: quick]: ").strip().lower() or "quick"
limit_pages = (mode == "quick")

print(f"Starting database build in {'Quick' if limit_pages else 'Full'} Mode...")
num_docs, num_chunks = build_index(
    csaf_dir="doc/cisa_csaf",
    csf_pdf_path="doc/NIST Cybersecurity Framework(CSF) 2.0.pdf",
    nist_pdf_path="doc/NIST.SP.800-82r3.pdf",
    db_path="chroma_db",
    collection_name="secureops_assistant",
    limit_pdf_pages=limit_pages,
    cve_csv_path="data/processed/cve_high_value.csv",
    cve_limit=2000,
    mitre_xlsx_path="doc/ics-attack-v19.1.xlsx"
)
print(f"Successfully indexed {num_chunks} chunks!")

## 🔍 Step 5: Run Q&A Queries

Ask cybersecurity questions to test the retriever, Cross-Encoder reranker, and generator.

In [ ]:
from src.retrieval import SecureOpsRetriever
from src.generation import SecureOpsGenerator

# Initialize components
retriever = SecureOpsRetriever(db_path="chroma_db", collection_name="secureops_assistant")
generator = SecureOpsGenerator()

def ask_assistant(query: str, vendor: str = None, severity: str = None):
    print(f"Query: {query}")
    if vendor or severity:
        print(f"Filters: vendor={vendor}, severity={severity}")
        
    # Retrieve context
    chunks = retriever.retrieve(query, k=5, vendor=vendor, severity=severity)
    
    # Generate answer
    answer, confidence, cited = generator.generate_answer(query, chunks)
    
    print(f"\nAnswer:\n{answer}\n")
    print(f"Confidence: {confidence * 100:.1f}%")
    
    if cited:
        print("\nCited Sources:")
        for idx, doc in enumerate(cited):
            meta = doc.get("metadata", {})
            print(f"  - [{idx+1}] {meta.get('source')} | Chapter: {meta.get('chapter', 'N/A')} | Section: {meta.get('section', 'N/A')} | Page: {meta.get('page_start', '?')}")
    print("=" * 80)

# Example 1: In-domain query
ask_assistant("What network segmentation recommendations does NIST SP 800-82 Rev 3 provide?")

In [ ]:
# Example 2: Query CISA CSAF Advisories (with filter)
ask_assistant("What default credential vulnerability affects MacGregor VDR devices?", vendor="Danelec")

In [ ]:
# Example 3: Out-of-domain query (Should trigger honest rejection)
ask_assistant("What is the CEO's personal phone number?")

## 📊 Step 6: Benchmark & Run Evaluation

Run the 36-case benchmark comparing dense retrieval with the full hybrid pipeline. This retrieval-only mode makes no paid LLM calls.

In [ ]:
from src.evaluate import run_evaluation
from IPython.display import Markdown, display

# Run evaluation
run_evaluation(qa_path="data/evaluation_qa.json", db_path="chroma_db", report_output="reports/evaluation_report.md", json_output="reports/evaluation_report.json")

# Display evaluation report
with open("reports/evaluation_report.md", "r", encoding="utf-8") as f:
    report_md = f.read()
    
display(Markdown(report_md))